# 01 — Data Cleaning

**Goal:** Load the raw UCI Online Retail transaction data and clean it into a form suitable
for RFM analysis — removing cancelled orders, missing customer IDs, and duplicates.

Output: `data/retail_clean.csv`

In [10]:
import sys
print(sys.executable)

c:\Users\DREAMUSER\Documents\customer-segmentation-clv\customer-segmentation-clv\venv\Scripts\python.exe


## 1. Load the Data

In [3]:
import pandas as pd
import numpy as np

# UCI provides this as .xlsx originally - if you downloaded the CSV version, use read_csv instead
df = pd.read_excel("../data/online_retail.xlsx")
# df = pd.read_csv("../data/online_retail.csv", encoding="ISO-8859-1")  # use this line if you have the CSV

df.columns = [c.strip() for c in df.columns]
print(f"Shape: {df.shape}")
df.head()

Shape: (541909, 8)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


## 2. Inspect Data Quality Issues

In [4]:
print("Missing values:")
print(df.isnull().sum())

print(f"\nNegative quantities (returns/cancellations): {(df['Quantity'] < 0).sum():,}")
print(f"Negative or zero unit prices: {(df['UnitPrice'] <= 0).sum():,}")
print(f"Duplicate rows: {df.duplicated().sum():,}")

Missing values:
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

Negative quantities (returns/cancellations): 10,624
Negative or zero unit prices: 2,517
Duplicate rows: 5,268


## 3. Remove Cancelled Orders and Invalid Rows

Invoice numbers starting with "C" indicate cancellations in this dataset. Negative quantities
and zero/negative prices don't represent real completed purchases, so they're excluded from
RFM analysis (which is about actual purchase behavior).

In [5]:
df["InvoiceNo"] = df["InvoiceNo"].astype(str)
cancellations = df["InvoiceNo"].str.startswith("C")
print(f"Cancelled invoices: {cancellations.sum():,} rows")

df = df[~cancellations]
df = df[df["Quantity"] > 0]
df = df[df["UnitPrice"] > 0]

print(f"Remaining rows after removing cancellations/invalid rows: {len(df):,}")

Cancelled invoices: 9,288 rows
Remaining rows after removing cancellations/invalid rows: 530,104


## 4. Remove Missing Customer IDs

In [6]:
missing_customer = df["CustomerID"].isnull().sum()
print(f"Rows with missing CustomerID: {missing_customer:,} ({missing_customer/len(df)*100:.1f}%)")

# these can't be attributed to any customer, so they're unusable for segmentation
df = df.dropna(subset=["CustomerID"])
df["CustomerID"] = df["CustomerID"].astype(int)

print(f"Remaining rows: {len(df):,}")
print(f"Unique customers: {df['CustomerID'].nunique():,}")

Rows with missing CustomerID: 132,220 (24.9%)
Remaining rows: 397,884
Unique customers: 4,338


## 5. Remove Duplicate Rows

In [7]:
before = len(df)
df = df.drop_duplicates()
print(f"Removed {before - len(df):,} duplicate rows")

Removed 5,192 duplicate rows


## 6. Compute Line Total and Fix Data Types

In [8]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
df["LineTotal"] = df["Quantity"] * df["UnitPrice"]

df = df.rename(columns={
    "InvoiceNo": "invoice_no",
    "StockCode": "stock_code",
    "Description": "description",
    "Quantity": "quantity",
    "InvoiceDate": "invoice_date",
    "UnitPrice": "unit_price",
    "CustomerID": "customer_id",
    "Country": "country",
    "LineTotal": "line_total",
})

df[["invoice_no", "customer_id", "invoice_date", "quantity", "unit_price", "line_total"]].head()

,invoice_no,customer_id,invoice_date,quantity,unit_price,line_total
0,536365,17850,2010-12-01 08:26:00,6,2.55,15.30
1,536365,17850,2010-12-01 08:26:00,6,3.39,20.34
2,536365,17850,2010-12-01 08:26:00,8,2.75,22.00
3,536365,17850,2010-12-01 08:26:00,6,3.39,20.34
4,536365,17850,2010-12-01 08:26:00,6,3.39,20.34


## 7. Save Cleaned Data

In [9]:
df.to_csv("../data/retail_clean.csv", index=False)
print(f"Saved {len(df):,} rows, {df['customer_id'].nunique():,} unique customers to "
      f"../data/retail_clean.csv")
print(f"Date range: {df['invoice_date'].min().date()} to {df['invoice_date'].max().date()}")

Saved 392,692 rows, 4,338 unique customers to ../data/retail_clean.csv
Date range: 2010-12-01 to 2011-12-09
